# Research data: inspect, pin and download

SDK 0.7. Default mode uses clearly labelled synthetic bytes, not a real Parquet file or FM. Download support.py beside this notebook. Explicit live mode inspects one available Iris catalogue entry and downloads its eligible pinned content without private workspace upload or fit. Downloaded bytes live in a disposable local directory; hosted state is never deleted.


In [ ]:
from support import notebook_research_client
from pathlib import Path
from tempfile import TemporaryDirectory
from hashlib import sha256

with notebook_research_client() as client:
    page = client.research_datasets(q="iris", availability="available", limit=10)
    if not page["items"]:
        raise RuntimeError("No eligible catalogue entry; no download or fallback.")
    selected = page["items"][0]["id"]
    info = client.research_dataset(selected)
    content = info["content"]
    if content is None:
        raise RuntimeError("Catalogue metadata is not download permission.")
    print(info["license"])
    suffix = {"application/vnd.apache.parquet": ".parquet", "application/zip": ".zip"}[content["media_type"]]
    with TemporaryDirectory(prefix="welt-research-example-") as folder:
        path = client.download_research_dataset(selected, Path(folder) / ("research" + suffix),
                                                version=content["version"])
        assert path.stat().st_size == content["size_bytes"]
        assert sha256(path.read_bytes()).hexdigest() == content["sha256"]
        print("Verified bytes; no private dataset created or model fitted.")


## Preserve the experiment contract

Inspect `schema`, `targets`, `splits`, `license` and `provenance` before using real bytes. Missing source information stays missing. Research download is separate from uploading a private execution table. A ZIP distribution includes `dataset.parquet`, `LICENSE.txt` and `ATTRIBUTION.txt`; retain all three, refuse unexpected members and never discard notices. Archive `content.sha256` differs from the inner table `provenance.canonical_sha256`. See the research guide for explicit safe extraction. The fixture never calls its labelled bytes valid Parquet. Full files can exceed model limits; choose and record an explicit split/seed/recipe rather than silently sampling.
